In [ ]:
import logging
import os
import threading
from importlib.resources import files
from pathlib import Path
import coraplex.orm.ormatic_interface # type: ignore
import numpy as np
import rclpy

from krrood.entity_query_language.verbalization.pipeline import verbalize_expression, VerbalizationPipeline
from krrood.ormatic.data_access_objects.helper import to_dao
from sqlalchemy.orm import sessionmaker
from semantic_digital_twin.world_description.world_entity import Body

from coraplex.datastructures.dataclasses import Context
from coraplex.datastructures.enums import Arms
from coraplex.execution_environment import simulated_robot
from coraplex.plans.factories import execute_single, sequential
from coraplex.robot_plans.actions.core.navigation import NavigateAction
from coraplex.robot_plans.actions.core.robot_body import ParkArmsAction
from krrood.entity_query_language.backends import ProbabilisticBackend, SQLAlchemyBackend
from krrood.entity_query_language.factories import *
from krrood.ormatic.utils import create_engine
from semantic_digital_twin.adapters.ros.tf_publisher import TFPublisher
from semantic_digital_twin.adapters.ros.visualization.spatial_type_marker_renderer import (
    SpatialTypeVisualization,
)
from semantic_digital_twin.adapters.ros.visualization.spatial_type_publisher import (
    SpatialTypePublisher,
)
from semantic_digital_twin.adapters.ros.visualization.viz_marker import (
    VizMarkerPublisher,
)
from semantic_digital_twin.adapters.urdf import URDFParser
from semantic_digital_twin.api import RobotSpecification
from semantic_digital_twin.reasoning.predicates import InsideOf
from semantic_digital_twin.reasoning.world_reasoner import WorldReasoner
from semantic_digital_twin.robots.pr2 import PR2
from semantic_digital_twin.semantic_annotations.mixins import IsStorageSpace
from semantic_digital_twin.semantic_annotations.semantic_annotations import Fridge, Milk
from semantic_digital_twin.semantic_annotations.semantic_annotations import ShelfLayer
from semantic_digital_twin.spatial_types import Point3, Quaternion
from semantic_digital_twin.spatial_types.spatial_types import (
    HomogeneousTransformationMatrix,
    Pose,
)
from semantic_digital_twin.world_description.geometry import Scale, Color
from semantic_digital_twin.world_description.graph_of_convex_sets.boxes import GraphOfBoundingBoxes
from semantic_digital_twin.world_description.graph_of_convex_sets.boxes import navigation_map_at_target
from semantic_digital_twin.world_description.world_entity import KinematicStructureEntity
from semantic_digital_twin.world_description.graph_of_convex_sets.base import translate_free_space_to_where_condition
from krrood.entity_query_language.verbalization.vocabulary.parts_of_speech import clause, Verb

from krrood.entity_query_language.verbalization.vocabulary.parts_of_speech import Noun
from krrood.entity_query_language.verbalization.fragments.base import VerbalizationFragment
from coraplex.plans.plan import Plan

from semantic_digital_twin.world import World


logging.disable(logging.CRITICAL)

rclpy.init()

node = rclpy.create_node("semantic_digital_twin")
thread = threading.Thread(target=rclpy.spin, args=(node,), daemon=True)
thread.start()

world_path = os.path.join(
    Path(files("coraplex")).parent.parent, "resources", "worlds", "kitchen-small.urdf"
)
world = URDFParser.from_file(world_path).parse()

SHELF_LAYER_SCALE = Scale(0.45, 0.5, 0.02)
"""
The extents of the shelf layer the milk stands on.

Fits into the fridge cavity, which is about 0.03 meters narrower than the shell on every
side.
"""

SHELF_LAYER_COLOR = Color(0.9, 0.93, 0.95)
"""
The colour of the shelf layer, an off-white against the fridge's own shell.
"""

FRIDGE_T_SHELF_LAYER = HomogeneousTransformationMatrix.from_xyz_rpy(
    0.0, 0.02, 0.0, yaw=np.pi
)
"""
The shelf layer in the fridge frame, centered in the cavity.

The kitchen places its fridge turned by half a turn against the room, so the layer turns
back: everything spawned below it is then aligned with the room, and grasped from the
front like any other object standing in it.
"""

SHELF_LAYER_T_MILK = HomogeneousTransformationMatrix.from_xyz_rpy(-0.16, 0.0, 0.11)
"""
The milk on the shelf layer, standing near its front edge.

Layer x runs towards the fridge opening, and the layer is 0.4 meters deep, so this
leaves the 0.065 meter carton just clear of the edge. The further forward it stands, the
further back the robot can stand to take it, and the less it has to lean into the swing
of the open door.
"""

MILK_NAME = "milk"
"""
Name of the transported body.
"""

SHELF_LAYER_NAME = "fridge_shelf"
"""
Name of the shelf layer the milk stands on.
"""

MILK_SCALE = Scale(0.065, 0.065, 0.2)
"""
The extents of the milk carton.
"""

tf_publisher = TFPublisher(_world=world, node=node)
viz = VizMarkerPublisher(_world=world, node=node)
points_publisher = SpatialTypePublisher(_world=world, node=node, topic_name="/semworld/sampled_points")

`WorldReasoner.reason()` runs a rule based reasoner over the world's kinematic structure and infers spatial relations between its bodies, such as which body is inside which container. The inferred relations are cached on the world and reused as long as the world does not change.

In [ ]:
world: World # <- this exists
WorldReasoner(world).reason()

In [ ]:
fridge = variable(Fridge, domain=world.semantic_annotations)
fridge_annotation = the(entity(fridge)).first()
shelf_layer = ShelfLayer.get_annotation_specification(
    SHELF_LAYER_NAME,
    ShelfLayer.get_default_root_kinematic_structure_entity_specification(
        scale=SHELF_LAYER_SCALE
    ),
).spawn(
    world,
    parent=fridge_annotation.root,
    parent_T_self=FRIDGE_T_SHELF_LAYER,
)
with world.modify_world():
    fridge_annotation.add(shelf_layer)
    for shape in shelf_layer.root.visual.shapes:
        shape.color = SHELF_LAYER_COLOR

Milk.get_annotation_specification(
    MILK_NAME,
    Milk.get_default_root_kinematic_structure_entity_specification(
        scale=MILK_SCALE
    ),
).spawn(world, parent=shelf_layer.root, parent_T_self=SHELF_LAYER_T_MILK)

A `symbolic_function` lets a plain Python function double as a building block for EQL queries. Called with concrete values it behaves like any function, but called with a `Variable` inside a query it defers execution instead, recording the call as a symbolic expression that is evaluated per candidate binding once the query runs.

**Your task:** implement `contains(container, body)` as a `symbolic_function`. It should decide whether `body` counts as being inside `container`, based on `InsideOf(body, container)()`, which returns the fraction of `body`'s volume that lies inside `container` (`1.0` fully contained, `0.0` not contained at all). Because that fraction rarely reaches exactly `1.0` for real meshes, treat `body` as contained once the fraction passes a high threshold, e.g. `0.9`. Give `contains` a docstring stating its contract, then try implementing it yourself before revealing the solution below.

In [ ]:
@symbolic_function
def contains(container: KinematicStructureEntity, body: KinematicStructureEntity) -> bool:
    """
    Decide whether ``body`` counts as being inside ``container``.

    :param container: the candidate storage space
    :param body: the body to test for containment
    :return: True if the fraction of ``body`` inside ``container`` exceeds a high threshold
    """
    ...

In [ ]:
@symbolic_function
def contains(container: KinematicStructureEntity, body: KinematicStructureEntity) -> bool:
    return InsideOf(body, container)() > 0.9

This query looks up the milk, then asks for the first `IsStorageSpace` annotation `c` whose root body `contains` it, using the `contains` symbolic function defined above as the query's `where` condition. In this kitchen that is the fridge, and from its `IsStorageSpace` annotation we take the handle of its first door: the target the robot needs to reach in order to open it.

In [ ]:
milk = the(Milk).first()
container = the(c := variable(IsStorageSpace)).where(contains(c.root, milk.root) ).first()
handle = container.doors[0].handle
print(type(container))

We now know *what* to fetch and *where* it is, the fridge and its handle, but not yet *how* to get there: where should the robot stand? Rather than picking a spot by hand, we phrase the question as a **generative query**: instead of filtering existing candidates with `where`, `a(...)` samples new values for a type's fields directly, drawing from any registered probabilistic backend.

**Your task:** write a generative query for a `Point3` positioned exactly at the handle's height but mirrored below it, with `x` and `y` left unconstrained, expressed relative to the handle's frame.

In [ ]:
backend = ProbabilisticBackend()

# TODO: generatively sample a Point3 at the handle's height, mirrored below it, with x
# and y left unconstrained, expressed relative to the handle's frame
plausible_location: Point3 = ...

plausible_location.expression.limit(1000)
results = list(plausible_location.evaluate(backend=backend))
points_publisher.set_requests(
    SpatialTypeVisualization(spatial_type=point, color=Color(0., 1., 0.))
    for point in results
)


In [ ]:
backend = ProbabilisticBackend()
plausible_location: Point3 = a(Point3)(x=..., y=..., z=-float(handle.root.global_pose.z), reference_frame=handle.root)
plausible_location.expression.limit(1000)
results = list(plausible_location.evaluate(backend= backend))
points_publisher.set_requests(
    SpatialTypeVisualization(spatial_type=point, color=Color(0., 1., 0.))
    for point in results
)

In RViz the sampled points are scattered all over the room, since `x` and `y` were left completely unconstrained, most of them are not places the robot could actually stand. Instead of sampling blindly, we now derive a better search space directly from the world's geometry: a map of the free space around the target, which we can then sample from or filter against.

In [ ]:
gbbs = navigation_map_at_target(handle.root, bloat_obstacles=0.5, search_range_x=5, search_range_y=5)
gbbs.plot_and_show_free_space()

**Your task:** implement `IsFree`, a predicate stating that a `Point3` lies within the navigable free space computed above. `__call__` should return whether `point` falls into one of `free_space`'s nodes, and `_verbalization_fragment_` should render the predicate as `<free_space> contains <point>`, the wording `verbalize_expression` and the pipeline above use to describe it. Method stubs are given below; fill them in yourself before revealing the solution.

In [ ]:
@dataclass
class IsFree(Predicate):
    """
    Predicate that holds when ``point`` lies in the free space ``free_space``.
    """

    free_space: GraphOfBoundingBoxes
    """The navigable free space to test the point against."""

    point: Point3
    """The point to test."""

    def __call__(self) -> bool:
        ...

    @classmethod
    def _verbalization_fragment_(cls, fields: RenderedFields) -> VerbalizationFragment:
        ...

backend = ProbabilisticBackend()

plausible_location: Point3 = a(Point3)(x=..., y=..., z=-float(handle.root.global_pose.z), reference_frame=handle.root)
plausible_location.expression.limit(1000)

plausible_location = a(Point3).from_(plausible_location.evaluate(backend=backend))
plausible_location.where(IsFree(gbbs, plausible_location.variable))

results = plausible_location.tolist()

points_publisher.set_requests(
    SpatialTypeVisualization(spatial_type=point, color=Color(0., 1., 0.))
    for point in results
)


In [ ]:
@dataclass
class IsFree(Predicate):
    free_space: GraphOfBoundingBoxes
    point: Point3

    def __call__(self) -> bool:
        return self.free_space.node_of_point(self.point) is not None

    @classmethod
    def _verbalization_fragment_(cls, fields: RenderedFields) -> VerbalizationFragment:
        return clause(
            Noun(fields["free_space"]),
            Verb("contains"),
            Noun(fields["point"]),
        )

backend = ProbabilisticBackend()

plausible_location: Point3 = a(Point3)(x=..., y=..., z=-float(handle.root.global_pose.z), reference_frame=handle.root)
plausible_location.expression.limit(1000)

plausible_location = a(Point3).from_(plausible_location.evaluate(backend=backend))
plausible_location.where(IsFree(gbbs, plausible_location.variable))

results = plausible_location.tolist()

points_publisher.set_requests(
    SpatialTypeVisualization(spatial_type=point, color=Color(0., 1., 0.))
    for point in results
)

In [ ]:
verbalize_expression(plausible_location)

In [ ]:
print(results[:10])
print(len(list(results)))

What we just did is **rejection sampling**: we drew points from an unconstrained distribution and then filtered out the ones that failed `IsFree`, discarding most of what we generated. We can do better by feeding the free space condition directly into the generating distribution instead of filtering after the fact, as long as the condition is written in disjunctive normal form (DNF) over variables the probabilistic model itself understands, via `translate_free_space_to_where_condition(gbbs.free_space_event, ...)`.

**Your task:** figure out which symbolic variable belongs in the second argument to `translate_free_space_to_where_condition`, the one the free space condition should constrain.

In [ ]:
plausible_location: Point3 = a(Point3)(x=..., y=..., z=-float(handle.root.global_pose.z), reference_frame=handle.root)
plausible_location.where(
    # TODO: which symbolic variable should the free space condition constrain?
    translate_free_space_to_where_condition(gbbs.free_space_event, ...))
plausible_location.expression.limit(1000)
VerbalizationPipeline.html(hierarchical=True).display(plausible_location)


In [ ]:
plausible_location: Point3 = a(Point3)(x=..., y=..., z=-float(handle.root.global_pose.z), reference_frame=handle.root)
plausible_location.where(
    translate_free_space_to_where_condition(gbbs.free_space_event, plausible_location.variable))
plausible_location.expression.limit(1000)
VerbalizationPipeline.html(hierarchical=True).display(plausible_location)

In [ ]:
results = list(plausible_location.evaluate(backend=backend))

points_publisher.set_requests(
    SpatialTypeVisualization(spatial_type=point, color=Color(0., 1., 0.))
    for point in results
)
print(results[:10])
print(len(list(results)))

In [ ]:
plausible_location: Point3 = a(Point3)(x=..., y=..., z=-float(handle.root.global_pose.z), reference_frame=handle.root)
plausible_location.where(
    translate_free_space_to_where_condition(gbbs.free_space_event, plausible_location.variable),
plausible_location.variable.x > 0)
plausible_location.expression.limit(1000)
VerbalizationPipeline.html(hierarchical=True).display(plausible_location)

In [ ]:
results = list(plausible_location.evaluate(backend=backend))

points_publisher.set_requests(
    SpatialTypeVisualization(spatial_type=point, color=Color(0., 1., 0.))
    for point in results
)

With a good place to stand, we can now move the robot there and get ready to open the fridge. This closes out the manipulation part of this exercise; what follows turns the plan we just executed into long term memory.

In [ ]:
points_publisher.clear()
robot = RobotSpecification(semantic_annotation_type=PR2).spawn(world)

navigation_target = Pose(position=results[0], orientation = Quaternion.from_rpy(roll=0., pitch=0, yaw=np.pi), reference_frame=handle.root)

context = Context.from_world(world)
plan = sequential([ParkArmsAction(Arms.BOTH),
            NavigateAction(target_location=navigation_target)], context=context).plan
with simulated_robot:
    plan.perform()

Everything we did so far may be worth remembering, so the robot can later learn from it. KRROOD lets us persist arbitrary objects, like the `plan` we just executed, into a database and query them again the same way we query the live world, so we set up a database held in memory and store the plan in it.

In [ ]:
session_maker = sessionmaker(bind=create_engine("sqlite:///:memory:"))
coraplex.orm.ormatic_interface.Base.metadata.create_all(bind=session_maker().bind)
session = session_maker()

long_term_memory = SQLAlchemyBackend(session_maker)

dao = to_dao(plan)

with long_term_memory.session_maker() as session:
    session.add(dao)
    session.commit()

The long term memory is just another EQL backend, so we can query it with the same `the(...)`/`a(...)` syntax used against the live world, here asking for the `Plan` we just stored.

In [ ]:
plan_query = the(Plan)
VerbalizationPipeline.html(hierarchical=True).display(plan_query, backend=long_term_memory)

In [ ]:
queried_plan = plan_query.tolist(backend=long_term_memory)
print(queried_plan)

**Your task:** write a query over the long term memory that returns every `Body` encountered during the experience.

In [ ]:
# TODO: query the long term memory for every Body encountered during the experience
body_query = ...
VerbalizationPipeline.html(hierarchical=True).display(body_query, backend=long_term_memory)


In [ ]:
body_query = a(Body)
VerbalizationPipeline.html(hierarchical=True).display(body_query, backend=long_term_memory)

In [ ]:
result = body_query.tolist(backend=long_term_memory)
print(result)

Finally, a stored plan is not just inspectable data: calling `from_dao()` reconstructs the full `Plan` object from long term memory, so it can be treated like working memory again and resumed from wherever it left off.

In [ ]:
reconstructed_plan: Plan = queried_plan[0].from_dao()
print(reconstructed_plan)